In [9]:
import numpy as np
import pandas as pd
from tqdm.auto import tqdm
from joblib import Parallel, delayed
from dgp_utils import generate_panel, generate_W_mcar, apply_W, get_conditional_mean_C0
from model_utils import fit_and_forecast_method
from eval_utils import evaluate_forecast, compare_methods_msfe, make_result_row, rows_to_dataframe, summarize_results_with_methods
from concurrent.futures import ProcessPoolExecutor
"""
import importlib
import eval_utils
importlib.reload(eval_utils)
"""


'\nimport importlib\nimport eval_utils\nimportlib.reload(eval_utils)\n'

In [ ]:
RANDOM_SEED = 123
rng = np.random.default_rng(RANDOM_SEED)

DGP_LIST = ["dgp1", "dgp2", "dgp3"]
METHOD_LIST = ["pca_var", "pca_lstm", "ae_var", "ae_lstm"]

N = 64
r = 2
H = 3
missing_rate = 0.30

T_train_list = [64, 128, 256]
n_reps = 50

## Setting hyperparameters

In [11]:
METHOD_KWARGS = {
    "pca_var": {
        "var_maxlags": 5,
        "var_ic": "bic",
    },
    "pca_lstm": {
        "lookback_L": 5,
        "lstm_hidden": 64,
        "lstm_epochs": 150,
        "lr": 1e-3,
    },
    "ae_var": {
        "ae_hidden": 128,
        "ae_epochs": 200,
        "lr": 1e-3,
        "var_maxlags": 5,
        "var_ic": "bic",
    },
    "ae_lstm": {
        "ae_hidden": 128,
        "lstm_hidden": 64,
        "ae_epochs": 200,
        "lstm_epochs": 150,
        "lookback_L": 5,
        "lr": 1e-3,
    },
}

In [12]:
def run_one_rep(dgp_name, method_name, T_train, rep, N=64, r=2, H=3, missing_rate=0.30):
    seed_base = 10000 + 1000 * rep + 100 * T_train

    T_total = T_train + H

    # Generate full panel and true latent path
    Y_full, z_full, params = generate_panel(
        dgp_name=dgp_name,
        T_total=T_total,
        N=N,
        r=r,
        seed=seed_base,
    )

    # Training part only
    Y_train = Y_full[:, :T_train]
    z_train = z_full[:T_train]

    # Missingness on training panel
    W = generate_W_mcar(Y_train, missing_rate=missing_rate, seed=seed_base + 1)
    Y_obs = apply_W(Y_train, W)

    # Forecast by chosen method
    method_out = fit_and_forecast_method(
        method_name=method_name,
        Y_obs=Y_obs,
        W=W,
        r=r,
        H=H,
        **METHOD_KWARGS[method_name],
    )

    # Oracle target C0
    C0 = get_conditional_mean_C0(
        z_train=z_train,
        params=params,
        H=H,
        n_mc=1000,
        seed=seed_base + 2,
    )

    # Evaluation
    eval_out = evaluate_forecast(
        Y_fore=method_out["Y_fore"],
        C0=C0,
        k_eval=32,
    )

    row = make_result_row(
        dgp_name=dgp_name,
        method_name=method_name,
        rep=rep,
        T_train=T_train,
        H=H,
        missing_rate=missing_rate,
        method_out=method_out,
        eval_out=eval_out,
    )

    return row, method_out, eval_out, C0

In [13]:
row, method_out, eval_out, C0 = run_one_rep(
    dgp_name="dgp1",
    method_name="pca_var",
    T_train=64,
    rep=0,
    N=N,
    r=r,
    H=H,
    missing_rate=missing_rate,
)

row

{'dgp': 'dgp1',
 'method': 'pca_var',
 'rep': 0,
 'T_train': 64,
 'H': 3,
 'missing_rate': 0.3,
 'k_eval': 32,
 'msfe_first_k': 0.0021070078949045074,
 'sd_sqerr_first_k': 0.00324360879119255,
 'factor_method': 'pca_missing',
 'dyn_method': 'var',
 'selected_lag': 1}

In [14]:
row2, _, _, _ = run_one_rep(
    dgp_name="dgp2",
    method_name="ae_lstm",
    T_train=64,
    rep=0,
    N=N,
    r=r,
    H=H,
    missing_rate=missing_rate,
)

row2

{'dgp': 'dgp2',
 'method': 'ae_lstm',
 'rep': 0,
 'T_train': 64,
 'H': 3,
 'missing_rate': 0.3,
 'k_eval': 32,
 'msfe_first_k': 0.022177505769225156,
 'sd_sqerr_first_k': 0.030612658821346412,
 'factor_method': 'ae',
 'dyn_method': 'lstm'}

In [7]:
rows = []

grid = [
    (dgp_name, method_name, T_train, rep)
    for dgp_name in DGP_LIST
    for method_name in METHOD_LIST
    for T_train in T_train_list
    for rep in range(n_reps)
]

for dgp_name, method_name, T_train, rep in tqdm(grid):
    row, _, _, _ = run_one_rep(
        dgp_name=dgp_name,
        method_name=method_name,
        T_train=T_train,
        rep=rep,
        N=N,
        r=r,
        H=H,
        missing_rate=missing_rate,
    )
    rows.append(row)

results_df = rows_to_dataframe(rows)
results_df.head()

100%|██████████| 720/720 [06:59<00:00,  1.72it/s]


,dgp,method,rep,T_train,H,missing_rate,k_eval,msfe_first_k,sd_sqerr_first_k,factor_method,dyn_method,selected_lag
0,dgp1,pca_var,0,64,3,0.3,32,0.002107,0.003244,pca_missing,var,1.0
1,dgp1,pca_var,1,64,3,0.3,32,0.004730,0.007490,pca_missing,var,1.0
2,dgp1,pca_var,2,64,3,0.3,32,0.057338,0.075920,pca_missing,var,1.0
3,dgp1,pca_var,3,64,3,0.3,32,0.005881,0.008234,pca_missing,var,1.0
4,dgp1,pca_var,4,64,3,0.3,32,0.009447,0.013702,pca_missing,var,1.0


In [ ]:
def run_grid_item(dgp_name, method_name, T_train, rep):
    row, _, _, _ = run_one_rep(
        dgp_name=dgp_name,
        method_name=method_name,
        T_train=T_train,
        rep=rep,
        N=N,
        r=r,
        H=H,
        missing_rate=missing_rate,
    )
    return row


grid = [
    (dgp_name, method_name, T_train, rep)
    for dgp_name in DGP_LIST
    for method_name in METHOD_LIST
    for T_train in T_train_list
    for rep in range(n_reps)
]

rows = Parallel(n_jobs=4)(
    delayed(run_grid_item)(dgp_name, method_name, T_train, rep)
    for dgp_name, method_name, T_train, rep in tqdm(grid)
)

results_df = rows_to_dataframe(rows)
results_df.head()

In [8]:
results_df.to_csv("ablation_results_raw.csv", index=False)
summary_df = summarize_results_with_methods(results_df)
summary_df
summary_df.to_csv("ablation_results_summary.csv", index=False)

In [18]:
cmp1 = compare_methods_msfe(results_df, "pca_var", "ae_var")
cmp2 = compare_methods_msfe(results_df, "pca_var", "pca_lstm")
cmp3 = compare_methods_msfe(results_df, "pca_var", "ae_lstm")

comparison_df = pd.concat([cmp1, cmp2, cmp3], ignore_index=True)
comparison_df.to_csv("ablation_results_pairwise_comparison_raw.csv", index=False)